In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse

from pandas.api.types import CategoricalDtype
from implicit.als import AlternatingLeastSquares
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score


## Задание 1. User-item матрица и train/test split


In [2]:
ratings = pd.read_csv('ratings.csv')

ratings.head()


,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3


In [3]:
print(f'Количество строк в ratings: {ratings.shape[0]}')
print(f'Количество уникальных пользователей: {ratings.user_id.nunique()}')
print(f'Количество уникальных книг: {ratings.book_id.nunique()}')

ratings['rating'].value_counts().sort_index()

Количество строк в ratings: 5976479
Количество уникальных пользователей: 53424
Количество уникальных книг: 10000


rating
1     124195
2     359257
3    1370916
4    2139018
5    1983093
Name: count, dtype: int64

In [4]:
# Для оценки релевантности считаем положительными оценки 4 и 5
# Оценки 1-3 остаются в ratings для разбиения истории пользователя,
# но не попадут в implicit user-item матрицу.
ratings['is_positive'] = ratings['rating'] >= 4

ratings['is_positive'].value_counts(normalize=True)

is_positive
True     0.689722
False    0.310278
Name: proportion, dtype: float64

In [5]:
# считаем текущий порядок строк внутри пользователя псевдохронологией
ratings = ratings.sort_values(['user_id']).copy()

ratings['book_order'] = ratings.groupby('user_id').cumcount() + 1
ratings['user_books_count'] = ratings.groupby('user_id')['book_id'].transform('count')
ratings['book_order_share'] = ratings['book_order'] / ratings['user_books_count']

ratings.head()

,user_id,book_id,rating,is_positive,book_order,user_books_count,book_order_share
0,1,258,5,True,1,117,0.008547
5704477,1,901,4,True,2,117,0.017094
5704479,1,212,3,False,3,117,0.025641
5704480,1,231,3,False,4,117,0.034188
1936422,1,128,5,True,5,117,0.042735


In [6]:
# Первые 70% истории каждого пользователя идут в train,
# последние 30% — в test
train_ratings = ratings[ratings['book_order_share'] <= 0.7].copy()
test_ratings = ratings[ratings['book_order_share'] > 0.7].copy()

# Для implicit-матрицы берём только положительные взаимодействия из train
train_positive = train_ratings[train_ratings['rating'] >= 4].copy()

print(f'Train size: {train_ratings.shape[0]}')
print(f'Test size: {test_ratings.shape[0]}')
print(f'Train positive size: {train_positive.shape[0]}')
print(f'Train users: {train_ratings.user_id.nunique()}')
print(f'Test users: {test_ratings.user_id.nunique()}')

Train size: 4159553
Test size: 1816926
Train positive size: 2868817
Train users: 53424
Test users: 53424


In [7]:
# Индексы строк и столбцов user-item матрицы
# Пользователей берём из train, книги — из полного каталога рейтингов
user_index = train_ratings['user_id'].unique()
book_index = ratings['book_id'].sort_values().unique()

user_cat = CategoricalDtype(categories=user_index)
book_cat = CategoricalDtype(categories=book_index)

rows = train_positive['user_id'].astype(user_cat).cat.codes
cols = train_positive['book_id'].astype(book_cat).cat.codes

print(rows.min(), rows.max())
print(cols.min(), cols.max())

0 53423
0 9999


In [8]:
# User-item матрица:
# строки — пользователи, столбцы — книги, значения — положительные взаимодействия
# В sparse-матрице отсутствующее значение означает отсутствие положительного взаимодействия
user_item_matrix = sparse.csr_matrix(
    (
        np.ones(len(train_positive)),
        (rows, cols)
    ),
    shape=(len(user_index), len(book_index))
)

print(f'Размер матрицы: {user_item_matrix.shape}')
print(f'Ненулевых элементов: {user_item_matrix.nnz}')
print(f'Положительных взаимодействий в train: {train_positive.shape[0]}')

Размер матрицы: (53424, 10000)
Ненулевых элементов: 2868817
Положительных взаимодействий в train: 2868817


In [9]:
# Выберем пользователя для дальнейших примеров
sample_user_id = train_positive['user_id'].iloc[0]
sample_user_idx = np.where(user_index == sample_user_id)[0][0]

print(f'sample_user_id: {sample_user_id}')
print(f'sample_user_idx: {sample_user_idx}')

sample_user_id: 1
sample_user_idx: 0


In [10]:
# Книги, которые выбранный пользователь положительно оценил в train
sample_user_books = train_ratings[
    (train_ratings['user_id'] == sample_user_id)
    & (train_ratings['rating'] >= 4)
][['user_id', 'book_id', 'rating']]

sample_user_books.head(20)

,user_id,book_id,rating
0,1,258,5
5704477,1,901,4
1936422,1,128,5
1936401,1,2770,4
1936424,1,81,5
1936403,1,198,4
1935750,1,1055,4
999469,1,46,4
1936419,1,1761,4
999515,1,131,4


**Вывод по заданию 1.**

Построена implicit user-item матрица по положительным взаимодействиям из train.
Оценки 1-3 исключены из матрицы.
Train/test split выполнен внутри каждого пользователя.


## Задание 2. Baseline, ALS и mAP@10


In [11]:
# Релевантные книги для оценки — книги из test с rating 4 или 5
test_positive = test_ratings[test_ratings['rating'] >= 4].copy()

test_relevant_by_user = (
    test_positive
    .groupby('user_id')['book_id']
    .apply(set)
    .to_dict()
)

# Книги, которые пользователь уже видел в train
# Исключаем их из рекомендаций, но test-книги оставляем для проверки попаданий
train_seen_by_user = (
    train_ratings
    .groupby('user_id')['book_id']
    .apply(set)
    .to_dict()
)

print(f'Users with positive test items: {len(test_relevant_by_user)}')

Users with positive test items: 53380


In [12]:
def apk(recommended_items, relevant_items, k=10):
    """Average Precision@K для одного пользователя."""
    if len(relevant_items) == 0:
        return 0.0

    recommended_items = recommended_items[:k]
    score = 0.0
    hits = 0

    for i, item in enumerate(recommended_items):
        if item in relevant_items:
            hits += 1
            score += hits / (i + 1)

    return score / min(len(relevant_items), k)

In [13]:
# Список пользователей для оценки: 500 случайных пользователей,
# у которых есть хотя бы одна положительная книга в test
candidate_users = list(test_relevant_by_user.keys())

np.random.seed(42)
eval_users = np.random.choice(candidate_users, size=500, replace=False)

eval_users[:10]

array([49145, 30950, 36047, 33111, 47046, 10711, 37923,  3931, 32671,
       13929])

In [14]:
# Random baseline: случайные книги, которых нет в train-истории пользователя
all_books = set(book_index)


def random_recommendations(user_id, n_recommendations=20):
    seen_books = train_seen_by_user.get(user_id, set())
    available_books = list(all_books - seen_books)
    n = min(n_recommendations, len(available_books))

    return list(
        np.random.choice(available_books, size=n, replace=False)
    )


random_ap_scores = []
np.random.seed(42)

for user_id in eval_users:
    recs = random_recommendations(user_id, n_recommendations=20)
    relevant_items = test_relevant_by_user.get(user_id, set())
    random_ap_scores.append(apk(recs, relevant_items, k=10))

random_map10 = np.mean(random_ap_scores)

print(f'Random baseline mAP@10: {random_map10:.6f}')

Random baseline mAP@10: 0.000860


In [15]:
# Popular baseline: самые популярные книги по положительным оценкам в train
popular_books = (
    train_positive
    .groupby('book_id')
    .size()
    .sort_values(ascending=False)
    .index
    .tolist()
)


def popular_recommendations(user_id, n_recommendations=20):
    seen_books = train_seen_by_user.get(user_id, set())
    recommendations = []

    for book_id in popular_books:
        if book_id not in seen_books:
            recommendations.append(book_id)

        if len(recommendations) == n_recommendations:
            break

    return recommendations


popular_ap_scores = []

for user_id in eval_users:
    recs = popular_recommendations(user_id, n_recommendations=20)
    relevant_items = test_relevant_by_user.get(user_id, set())
    popular_ap_scores.append(apk(recs, relevant_items, k=10))

popular_map10 = np.mean(popular_ap_scores)

print(f'Popular baseline mAP@10: {popular_map10:.6f}')

Popular baseline mAP@10: 0.059875


In [16]:
# Базовая ALS-модель
als_model = AlternatingLeastSquares(
    factors=64,
    iterations=20,
    regularization=0.01,
    random_state=42
)

als_model.fit(user_item_matrix)

C:\ProgramData\anaconda3\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 24 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

In [17]:
user_id_to_idx = {user_id: idx for idx, user_id in enumerate(user_index)}


def als_recommendations(model, user_id, n_recommendations=20):
    if user_id not in user_id_to_idx:
        return []

    user_idx = user_id_to_idx[user_id]

    item_ids, scores = model.recommend(
        userid=user_idx,
        user_items=user_item_matrix[user_idx],
        N=n_recommendations,
        filter_already_liked_items=True
    )

    return [book_index[item_id] for item_id in item_ids]


def evaluate_model_map10(recommend_func, users):
    ap_scores = []

    for user_id in users:
        recs = recommend_func(user_id)
        relevant_items = test_relevant_by_user.get(user_id, set())
        ap_scores.append(apk(recs, relevant_items, k=10))

    return np.mean(ap_scores)

In [18]:
als_map10 = evaluate_model_map10(
    lambda user_id: als_recommendations(als_model, user_id, n_recommendations=20),
    eval_users
)

print(f'Random baseline mAP@10:  {random_map10:.6f}')
print(f'Popular baseline mAP@10: {popular_map10:.6f}')
print(f'ALS mAP@10:              {als_map10:.6f}')

if als_map10 > popular_map10:
    print('ALS превзошёл popular baseline.')
else:
    print('ALS не превзошёл popular baseline.')

Random baseline mAP@10:  0.000860
Popular baseline mAP@10: 0.059875
ALS mAP@10:              0.138668
ALS превзошёл popular baseline.


In [19]:
# Рекомендации ALS для выбранного пользователя
sample_als_recs = als_recommendations(
    als_model,
    sample_user_id,
    n_recommendations=10
)

pd.DataFrame({'book_id': sample_als_recs})

,book_id
0,10
1,94
2,172
3,118
4,57
5,55
6,85
7,80
8,150
9,35


**Вывод по заданию 2.**

Реализована метрика AP@K, посчитаны random и popular baseline, обучена ALS-модель.
Качество ALS сравнивается с baseline на одних и тех же 500 пользователях.


## Задание 3. Применение комбинации методов, подсчёт метрик


In [20]:
# Признаки пользователей из train
user_features = (
    train_ratings
    .groupby('user_id')
    .agg(
        user_rating_count=('book_id', 'count'),
        user_mean_rating=('rating', 'mean'),
        user_positive_count=('is_positive', 'sum'),
    )
    .reset_index()
)

user_features['user_positive_rate'] = (
    user_features['user_positive_count'] / user_features['user_rating_count']
)

user_features.to_csv('user_features.csv', index=False)

user_features.head()

,user_id,user_rating_count,user_mean_rating,user_positive_count,user_positive_rate
0,1,81,3.469136,38,0.469136
1,2,45,4.533333,38,0.844444
2,3,63,1.698413,2,0.031746
3,4,93,3.752688,59,0.634409
4,5,70,4.114286,59,0.842857


In [21]:
# Признаки книг из train
book_features = (
    train_ratings
    .groupby('book_id')
    .agg(
        book_rating_count=('user_id', 'count'),
        book_mean_rating=('rating', 'mean'),
        book_positive_count=('is_positive', 'sum'),
    )
    .reset_index()
)

book_features['book_positive_rate'] = (
    book_features['book_positive_count'] / book_features['book_rating_count']
)

book_features.to_csv('book_features.csv', index=False)

book_features.head()

,book_id,book_rating_count,book_mean_rating,book_positive_count,book_positive_rate
0,1,15691,4.275636,13014,0.829393
1,2,15188,4.354095,12737,0.838623
2,3,11846,3.211886,5398,0.455681
3,4,13386,4.328478,11255,0.840804
4,5,11648,3.767943,7344,0.630495


In [22]:
feature_cols = [
    'user_rating_count',
    'user_mean_rating',
    'user_positive_count',
    'user_positive_rate',
    'book_rating_count',
    'book_mean_rating',
    'book_positive_count',
    'book_positive_rate',
]


def make_feature_table(pairs_df):
    features = (
        pairs_df
        .merge(user_features, on='user_id', how='left')
        .merge(book_features, on='book_id', how='left')
    )

    features[feature_cols] = features[feature_cols].fillna(0)

    return features

In [23]:
# Обучающая выборка для классификатора:
# берём положительные и отрицательные оценки из train
positive_train = train_ratings[train_ratings['rating'] >= 4].copy()
negative_train = train_ratings[train_ratings['rating'] < 4].copy()

negative_sample = negative_train.sample(
    n=min(len(negative_train), len(positive_train)),
    random_state=42
)

clf_train = pd.concat([positive_train, negative_sample], ignore_index=True)
clf_train['target'] = (clf_train['rating'] >= 4).astype(int)

clf_train_features = make_feature_table(
    clf_train[['user_id', 'book_id', 'target']]
)

clf_train_features.head()

,user_id,book_id,target,user_rating_count,user_mean_rating,user_positive_count,user_positive_rate,book_rating_count,book_mean_rating,book_positive_count,book_positive_rate
0,1,258,1,81,3.469136,38,0.469136,2490,4.128916,1916,0.769478
1,1,901,1,81,3.469136,38,0.469136,1035,4.000966,773,0.746860
2,1,128,1,81,3.469136,38,0.469136,1174,4.096252,916,0.780239
3,1,2770,1,81,3.469136,38,0.469136,307,4.016287,219,0.713355
4,1,81,1,81,3.469136,38,0.469136,4044,4.136004,3225,0.797478


In [24]:
reranker = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

reranker.fit(
    clf_train_features[feature_cols],
    clf_train_features['target']
)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total 

In [25]:
def hybrid_recommendations(user_id, n_candidates=30, n_recommendations=10):
    # 1. Получаем кандидатов из ALS
    candidates = als_recommendations(
        als_model,
        user_id,
        n_recommendations=n_candidates
    )

    if len(candidates) == 0:
        return []

    # 2. Собираем признаки user-book для кандидатов
    candidate_df = pd.DataFrame({
        'user_id': [user_id] * len(candidates),
        'book_id': candidates,
    })

    candidate_features = make_feature_table(candidate_df)

    # 3. Классификатор оценивает вероятность положительной оценки
    proba = reranker.predict_proba(candidate_features[feature_cols])[:, 1]
    candidate_features['score'] = proba

    # 4. Сортируем кандидатов по score и берём top-N
    reranked = (
        candidate_features
        .sort_values('score', ascending=False)
        ['book_id']
        .head(n_recommendations)
        .tolist()
    )

    return reranked

In [26]:
hybrid_map10 = evaluate_model_map10(
    lambda user_id: hybrid_recommendations(
        user_id,
        n_candidates=30,
        n_recommendations=10
    ),
    eval_users
)

print(f'Popular baseline mAP@10: {popular_map10:.6f}')
print(f'ALS mAP@10:              {als_map10:.6f}')
print(f'Hybrid mAP@10:           {hybrid_map10:.6f}')

Popular baseline mAP@10: 0.059875
ALS mAP@10:              0.138668
Hybrid mAP@10:           0.084021


In [27]:
sample_hybrid_recs = hybrid_recommendations(
    sample_user_id,
    n_candidates=30,
    n_recommendations=10
)

pd.DataFrame({'book_id': sample_hybrid_recs})

,book_id
0,192
1,189
2,85
3,291
4,265
5,103
6,10
7,23
8,80
9,325


**Вывод по заданию 3.**

В задании 3 был реализован гибридный подход к рекомендациям. Сначала ALS использовался как генератор кандидатов: для каждого пользователя модель отбирала до 30 потенциально подходящих книг. Затем эти кандидаты дополнительно ранжировались классификатором RandomForestClassifier.

Для обучения классификатора использовались положительные и отрицательные примеры из train-части. Положительным классом считались оценки 4 и 5, отрицательным — оценки ниже 4. Чтобы классы были сбалансированы, отрицательные примеры были взяты сэмплом в количестве, сопоставимом с числом положительных.
После обучения классификатор применялся к ALS-кандидатам и сортировал их по вероятности положительной оценки.
Полученные результаты:

Popular baseline mAP@10: 0.059875

ALS mAP@10:              0.138668

Hybrid mAP@10:           0.084021

Гибридная модель превзошла baseline популярных книг. Однако гибридная модель оказалась хуже обычного ALS. То есть в текущей реализации ALS сам по себе ранжирует рекомендации лучше, чем связка ALS candidates + RandomForest reranking.